In [7]:
import torch
import pandas as pl # Using polars or pandas
import numpy as np

def compute_final_fusion_score(preds_dict, weights, fatigue_penalties=None):
    """
    Computes the final ranking score using a non-linear utility function.
    
    Args:
        preds_dict (dict): Dictionary of PyTorch tensors containing predictions
                           for 'p_click', 'p_like', 'p_comment', 'p_forward', 
                           'p_hate', 'p_long_view', 'watch_time'.
        weights (dict): Hyperparameters alpha, beta, gamma, delta, lambda.
        fatigue_penalties (Tensor, optional): Author/Topic fatigue penalties per item.
        
    Returns:
        Tensor: A 1D tensor of Final Scores to sort the candidates.
    """
    # 1. Base Multiplicative Engagement Boost
    # If the user clicks, how much extra value do we get if they also Like or Share?
    engagement_boost = (
        1.0 + 
        (weights['alpha_like'] * preds_dict['p_like']) + 
        (weights['alpha_comment'] * preds_dict['p_comment']) + 
        (weights['beta_share'] * preds_dict['p_forward']) +
        (weights['alpha_long_view'] * preds_dict['p_long_view'])
    )
    
    # 2. Continuous Watch Time Scaling (Gamma)
    # We use a power-law to gently reward longer watch times without letting 
    # a 10-minute video automatically beat a viral 15-second video.
    # Note: watch_time is in ms, we scale it to seconds for numerical stability.
    watch_time_seconds = preds_dict['watch_time'] / 1000.0
    time_factor = torch.pow(watch_time_seconds + 1.0, weights['gamma_time'])
    
    # 3. Negative Suppression (Delta)
    # If p_hate is high, this term rapidly approaches 0, killing the final score.
    suppression_factor = torch.pow(1.0 - preds_dict['p_hate'], weights['delta_hate'])
    
    # 4. Master Equation
    # Final Score = p_click * (Engagement Boost) * (Watch Time Factor) * (Suppression)
    raw_scores = (
        preds_dict['p_click'] * engagement_boost * time_factor * suppression_factor
    )
    
    # 5. Apply Session Fatigue Penalty (Lambda)
    # Suppress videos from creators the user just saw 3 times in a row.
    if fatigue_penalties is not None:
        final_scores = raw_scores - (weights['lambda_fatigue'] * fatigue_penalties)
    else:
        final_scores = raw_scores
        
    return final_scores


def assemble_final_feed(candidate_item_ids, mmoe_predictions, author_ids):
    """
    Simulates the final rendering step. Sorts the 200 candidates and applies
    a lightweight Author Diversity filter to return the top 10 for the mobile UI.
    """
    # Define business-logic weights (These are normally tuned via A/B testing)
    business_weights = {
        'alpha_like': 2.0,      # Likes are worth 2x a base click
        'alpha_comment': 1.5,
        'beta_share': 5.0,      # Shares are highly viral, worth 5x
        'alpha_long_view': 1.0, 
        'gamma_time': 0.3,      # Gentle sub-linear curve for watch time
        'delta_hate': 3.0,      # Strong aggressive penalty for skip/hate
        'lambda_fatigue': 0.5
    }
    
    # Simulate calculating a fatigue penalty (e.g., if author was recently seen)
    # In reality, this checks a Redis cache of the user's current session.
    current_device = mmoe_predictions['p_click'].device
    fatigue_penalties = torch.zeros(len(candidate_item_ids), device=current_device) 
    
    # 1. Compute single scalar score
    final_scores = compute_final_fusion_score(mmoe_predictions, business_weights, fatigue_penalties)
    
    # 2. Sort candidates by Final Score descending
    sorted_indices = torch.argsort(final_scores, descending=True)
    
    # 3. Post-Ranking Filter (Author Diversity)
    # Ensure no two adjacent videos in the final feed are from the same creator
    final_feed_ids = []
    seen_authors = set()
    
    for idx in sorted_indices.tolist():
        vid = candidate_item_ids[idx]
        author = author_ids[idx]
        
        # Diversity check: Skip if we just showed this author
        if author in seen_authors:
            continue
            
        final_feed_ids.append(vid)
        seen_authors.add(author)
        
        # Stop once we have 10 videos to send to the client
        if len(final_feed_ids) >= 10:
            break
            
    return final_feed_ids

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class TargetAttention(nn.Module):
    """Deep Interest Network (DIN) Target Attention Module"""
    def __init__(self, embed_dim=32):
        super().__init__()
        self.attn_mlp = nn.Sequential(
            nn.Linear(embed_dim * 4, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, target_item_emb, history_seq_embs):
        # target_item_emb: (Batch, Dim)
        # history_seq_embs: (Batch, SeqLen, Dim)
        seq_len = history_seq_embs.size(1)
        target_expanded = target_item_emb.unsqueeze(1).expand(-1, seq_len, -1)
        
        # Combine [Target, Sequence Item, Target - Sequence Item, Target * Sequence Item]
        concat_feat = torch.cat([
            target_expanded,
            history_seq_embs,
            target_expanded - history_seq_embs,
            target_expanded * history_seq_embs
        ], dim=-1)
        
        attn_weights = self.attn_mlp(concat_feat) # (Batch, SeqLen, 1)
        attn_weights = F.softmax(attn_weights, dim=1)
        
        # Weighted sum of past user history vectors
        user_interest_vector = torch.sum(attn_weights * history_seq_embs, dim=1)
        return user_interest_vector


class MMoELayer(nn.Module):
    """Multi-gate Mixture-of-Experts (MMoE) Layer"""
    def __init__(self, input_dim, num_experts=4, expert_hidden_dim=128, num_tasks=7):
        super().__init__()
        self.num_experts = num_experts
        self.num_tasks = num_tasks
        
        # 2. Two-Layer Experts: Added a second linear transformation and activation.
        # This increases the capacity of each expert to model nonlinear feature relationships.
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, expert_hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(expert_hidden_dim, expert_hidden_dim), # Second Layer
                nn.ReLU()
            ) for _ in range(num_experts)
        ])
        
        # Task Gating Networks
        self.gates = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, num_experts),
                nn.Softmax(dim=-1)
            ) for _ in range(num_tasks)
        ])

    def forward(self, x):
        # x shape: (Batch, input_dim)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1) # (Batch, NumExperts, HiddenDim)
        
        task_inputs = []
        for gate in self.gates:
            gate_weights = gate(x).unsqueeze(-1) # (Batch, NumExperts, 1)
            task_representation = torch.sum(gate_weights * expert_outputs, dim=1) # (Batch, HiddenDim)
            task_inputs.append(task_representation)
            
        return task_inputs


class MMoERankingModel(nn.Module):
    def __init__(self, num_items, num_categories, num_tags, item_embed_dim=32, nlp_embed_dim=512):
        super().__init__()
        
        # Embeddings
        self.item_embedding = nn.Embedding(num_items, item_embed_dim, padding_idx=0)
        self.cat1_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.cat2_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.tag_embedding  = nn.Embedding(num_tags, 8, padding_idx=0)
        
        self.target_attention = TargetAttention(embed_dim=item_embed_dim)
        
        # NLP Reduction MLP
        self.text_mlp = nn.Sequential(nn.Linear(nlp_embed_dim, 32), nn.ReLU())
        
        # Original Input Dimension: 150
        raw_input_dim = 32 + 24 + 32 + 8 + 8 + 8 + 1 + 5 + 32
        
        # 1. Feature Preprocessing MLP:
        # We pass the raw concatenated features through this MLP before hitting the MMoE.
        # This lets distinct features interact globally before being routed to experts.
        mmoe_input_dim = 128
        self.feature_preprocessing = nn.Sequential(
            nn.Linear(raw_input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, mmoe_input_dim),
            nn.ReLU()
        )
        
        # MMoE Engine (Now receives the 128-dim preprocessed features)
        self.mmoe = MMoELayer(input_dim=mmoe_input_dim, num_experts=4, expert_hidden_dim=64, num_tasks=7)
        
        # 3. Two-Layer Task Towers:
        # A helper function builds deeper task-specific heads instead of single linear layers.
        def build_task_tower(hidden_dim):
            return nn.Sequential(
                nn.Linear(hidden_dim, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            )

        # 7 Multi-Task Prediction Heads
        self.head_click = build_task_tower(64)       # Task 1: pCTR
        self.head_like = build_task_tower(64)        # Task 2: pLike
        self.head_comment = build_task_tower(64)     # Task 3: pComment
        self.head_forward = build_task_tower(64)     # Task 4: pForward
        self.head_hate = build_task_tower(64)        # Task 5: pHate
        self.head_long_view = build_task_tower(64)   # Task 6: pFinish / pLongView
        self.head_watch_time = build_task_tower(64)  # Task 7: Expected Watch Time

    def forward(self, history_seq, user_static, item_id, cat1_id, cat2_id, tags, item_stats, dur_log, nlp_vec):
        # Item Embeddings
        target_item_emb = self.item_embedding(item_id)
        history_seq_embs = self.item_embedding(history_seq)
        
        # Target Attention Over History
        user_interest = self.target_attention(target_item_emb, history_seq_embs)
        
        c1_emb = self.cat1_embedding(cat1_id)
        c2_emb = self.cat2_embedding(cat2_id)
        tag_emb = self.tag_embedding(tags).sum(dim=1)
        text_feat = self.text_mlp(nlp_vec)
        dur_feat = dur_log.unsqueeze(1)
        
        # Concatenate ALL User, Context, and Item features into a Single Representation
        dense_concat = torch.cat([
            user_interest, user_static, target_item_emb, 
            c1_emb, c2_emb, tag_emb, dur_feat, item_stats, text_feat
        ], dim=-1)
        
        # 1. Pass through Feature Preprocessing MLP
        preprocessed_features = self.feature_preprocessing(dense_concat)
        
        # MMoE Forward Pass
        task_representations = self.mmoe(preprocessed_features)
        
        # 4. Remove Sigmoids: BCEWithLogitsLoss requires raw logits!
        # 5. Remove F.relu on watch_time: We are predicting log-transformed watch time, 
        #    which can theoretically be modeled as a continuous real number.
        outputs = {
            "p_click_logits": self.head_click(task_representations[0]),
            "p_like_logits": self.head_like(task_representations[1]),
            "p_comment_logits": self.head_comment(task_representations[2]),
            "p_forward_logits": self.head_forward(task_representations[3]),
            "p_hate_logits": self.head_hate(task_representations[4]),
            "p_long_view_logits": self.head_long_view(task_representations[5]),
            "watch_time_log": self.head_watch_time(task_representations[6]) 
        }
        return outputs

class MultiTaskLoss(nn.Module):
    # Notice the watch_time weight is adjusted up slightly since log(ms) shrinks the loss scale.
    def __init__(self, weights={"click": 1.0, "like": 2.0, "comment": 2.0, "forward": 3.0, "hate": 3.0, "long_view": 1.5, "watch_time_log": 0.5}):
        super().__init__()
        # 4. Replace BCELoss with BCEWithLogitsLoss for far better numerical stability
        self.bce_logits = nn.BCEWithLogitsLoss()
        self.mse = nn.MSELoss()
        self.w = weights

    def forward(self, preds, targets):
        # Calculate BCE loss using raw un-sigmoid'd logits
        l_click = self.bce_logits(preds["p_click_logits"].squeeze(), targets["is_click"].float())
        l_like = self.bce_logits(preds["p_like_logits"].squeeze(), targets["is_like"].float())
        l_comment = self.bce_logits(preds["p_comment_logits"].squeeze(), targets["is_comment"].float())
        l_forward = self.bce_logits(preds["p_forward_logits"].squeeze(), targets["is_forward"].float())
        l_hate = self.bce_logits(preds["p_hate_logits"].squeeze(), targets["is_hate"].float())
        l_long_view = self.bce_logits(preds["p_long_view_logits"].squeeze(), targets["long_view"].float())
        
        # 5. Predict Log-Transformed Watch Time. 
        # We use torch.log1p (log(1 + x)) on the targets to safely avoid log(0)
        target_watch_time_log = torch.log1p(targets["play_time_ms"].float())
        l_watch_time = self.mse(preds["watch_time_log"].squeeze(), target_watch_time_log)
        
        total_loss = (
            self.w["click"] * l_click +
            self.w["like"] * l_like +
            self.w["comment"] * l_comment +
            self.w["forward"] * l_forward +
            self.w["hate"] * l_hate +
            self.w["long_view"] * l_long_view +
            self.w["watch_time_log"] * l_watch_time
        )
        return total_loss

In [9]:
!pip install polars

In [11]:
import os
import time
import torch
import polars as pl
import numpy as np

# ==========================================
# 1. INITIALIZATION (Modified for GPU & Lazy Data)
# ==========================================

def initialize_mmoe_pipeline(num_items=4000000, num_categories=250, num_tags=50000):
    """
    Initializes the MMoE ranking model for GPU.
    We pass the bounds directly to avoid passing massive eager dataframes.
    """
    print(f"📊 Vocab Sizes: Items={num_items}, Categories={num_categories}, Tags={num_tags}")

    # --- 1. Target the GPU Device ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"💻 Utilizing hardware device: {device}")

    # --- 2. Initialize Model ---
    # (Assuming MMoERankingModel and MultiTaskLoss are defined above this)
    model = MMoERankingModel(
        num_items=num_items,
        num_categories=num_categories,
        num_tags=num_tags,
        item_embed_dim=32,
        nlp_embed_dim=512 
    ).to(device)

    # --- 3. Initialize Loss & Optimizer ---
    criterion = MultiTaskLoss(
        weights={
            "click": 1.0, 
            "like": 2.0, 
            "comment": 2.0, 
            "forward": 3.0, 
            "hate": 3.0, 
            "long_view": 1.5, 
            "watch_time_log": 0.5
        }
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    return model, criterion, optimizer, device

# ==========================================
# 2. INFERENCE PIPELINE
# ==========================================

def run_ranking_inference():
    print("📥 Lazy-Loading KuaiRand Datasets (Preventing OOM)...")
    
    # Use scan_parquet (LazyFrame) instead of read_parquet (DataFrame)
    # This prevents loading the entire multi-gigabyte file into RAM.
    q_items = pl.scan_parquet("/kaggle/input/datasets/nguyenngocanhle/mtl-data/ranking_item_table.parquet")

    # Fetch ONLY the first 200 candidate items into memory
    print("🔎 Extracting Top 200 Candidates...")
    candidates_df = q_items.head(200).collect()
    candidate_item_ids = candidates_df['video_id'].to_list()
    
    if 'author_id' in candidates_df.columns:
        author_ids = candidates_df['author_id'].to_list()
    else:
        author_ids = [f"author_{i%50}" for i in range(200)] 
        
    # --- MODEL LOADING & DEVICE TARGETING ---
    model_path = "/kaggle/input/models/nguyenngocanhle/kuairand-1k-mmoe/pytorch/default/2/kaggle/working/best_mmoe_ranking_tpu.pth"
    
    # Initialize pipeline directly onto the GPU
    model, _, _, device = initialize_mmoe_pipeline()
    
    if os.path.exists(model_path):
        print(f"✅ Found weights at: {model_path}")
        # Standard PyTorch GPU loading 
        # model.load_state_dict(torch.load(model_path, map_location=device))
        # model.eval()
    else:
        print(f"⚠️ Model path not found. Generating simulated predictions on GPU for testing.")
        
    # Simulated predictions directly on the GPU
    mmoe_predictions = {
        'p_click': torch.rand(200, device=device),
        'p_like': (torch.rand(200, device=device) * 0.15),     
        'p_comment': (torch.rand(200, device=device) * 0.05),  
        'p_forward': (torch.rand(200, device=device) * 0.02),  
        'p_hate': (torch.rand(200, device=device) * 0.08),     
        'p_long_view': torch.rand(200, device=device),
        'watch_time': (torch.rand(200, device=device) * 45000) 
    }

    # ==========================================
    # 3. LATENCY MEASUREMENT & FINAL RANKING1
    # ==========================================
    print("\n⚡ Executing Final Score Fusion & Sorting on GPU...")
    
    # Sync before starting the timer to ensure accurate measurement
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()
    
    # Run the fusion pipeline
    final_feed = assemble_final_feed(candidate_item_ids, mmoe_predictions, author_ids)
    
    # Sync again before stopping the clock to ensure the GPU has actually finished 
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()
    
    latency_ms = (end_time - start_time) * 1000

    print(f"\n--- 🎬 Ranking Complete ---")
    print(f"Return Feed Size : {len(final_feed)} videos")
    print(f"Top 10 Video IDs : {final_feed}")
    print(f"⏱️ GPU Latency    : {latency_ms:.3f} ms")

run_ranking_inference()

📥 Lazy-Loading KuaiRand Datasets (Preventing OOM)...
🔎 Extracting Top 200 Candidates...
📊 Vocab Sizes: Items=4000000, Categories=250, Tags=50000
💻 Utilizing hardware device: cuda
✅ Found weights at: /kaggle/input/models/nguyenngocanhle/kuairand-1k-mmoe/pytorch/default/2/kaggle/working/best_mmoe_ranking_tpu.pth

⚡ Executing Final Score Fusion & Sorting on GPU...

--- 🎬 Ranking Complete ---
Return Feed Size : 10 videos
Top 10 Video IDs : [3927009, 4328273, 4239347, 3661880, 3691416, 4254854, 4127356, 3920123, 3688558, 3748193]
⏱️ GPU Latency    : 0.655 ms
